In [ ]:
import os
import json

file_path = r"C:\project\political_ner\output_results.json"

with open(file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

In [ ]:
import pandas as pd

file_path = r'C:\project\political_ner\Glossary\NER_spain_glossary.xlsx'
excel_file = pd.ExcelFile(file_path)
print("Sheet Names:", excel_file.sheet_names)

df = pd.read_excel(file_path, sheet_name='Glossary')

In [ ]:
import json
import pandas as pd

file_path_json = r"C:\project\political_ner\output_results.json"
with open(file_path_json, 'r', encoding='utf-8') as file:
    data = json.load(file)

replace_dict = {}
for entry in data:
    corrected = entry["CorrectedName"]
    for orig in entry["Original"]:
        replace_dict[orig] = corrected

file_path_excel = r'C:\project\political_ner\Glossary\NER_spain_glossary.xlsx'
df = pd.read_excel(file_path_excel, sheet_name='Glossary')

df["Entry"] = df["Entry"].replace(replace_dict)

df.drop_duplicates(subset=["Entry"], keep="first", inplace=True)

output_file = r'C:\project\political_ner\cleaned_file.xlsx'
df.to_excel(output_file, index=False)


In [ ]:
file_path_excel = r'C:\project\political_ner\cleaned_file.xlsx'
clean_df = pd.read_excel(file_path_excel)

In [ ]:
clean_df

In [ ]:
import json
import pandas as pd
from thefuzz import process  # or from fuzzywuzzy import process


clean_df = df  

# Build a single reference list of all possible strings (Entry + each Variation).
# Also build a lookup dict to map each variation string back to the canonical Entry.
all_strings = set()
variation_to_entry_map = {}

for _, row in clean_df.iterrows():
    canonical_entry = row["Entry"]
    
    # 1) Add the canonical entry as a possible match.
    all_strings.add(canonical_entry)
    variation_to_entry_map[canonical_entry] = canonical_entry
    
    # 2) Add each variation.
    if pd.notna(row["Variations"]):
        # If variations are comma-separated, split them
        variations_list = [v.strip() for v in row["Variations"].split(",")]
        for var in variations_list:
            if var:
                all_strings.add(var)
                variation_to_entry_map[var] = canonical_entry

all_strings_list = list(all_strings)

# ------------------------------------------------------------------
# 2. Load your second JSON, which has "Original", "Persons", "Parties"
# ------------------------------------------------------------------
json_path = r"C:\project\political_ner\output_results.json"
with open(json_path, 'r', encoding='utf-8') as f:
    data_json = json.load(f)

# ------------------------------------------------------------------
# 3. Fuzzy-match "Persons" and "Parties" in each record
# ------------------------------------------------------------------
THRESHOLD = 60  # minimum score you consider "good enough" (tweak as needed)

for item in data_json:
    # Initialize the new keys
    item["corrected_persons"] = ""
    item["corrected_parties"] = ""
    
    # 3a. If "Persons" is not empty, fuzzy match
    if item.get("Persons"):
        best_match, best_score = process.extractOne(item["Persons"], all_strings_list)
        if best_match and best_score >= THRESHOLD:
            # Map fuzzy-match result back to the canonical Entry
            canonical_entry = variation_to_entry_map[best_match]
            item["corrected_persons"] = canonical_entry
    
    # 3b. If "Parties" is not empty, fuzzy match
    if item.get("Parties"):
        best_match, best_score = process.extractOne(item["Parties"], all_strings_list)
        if best_match and best_score >= THRESHOLD:
            canonical_entry = variation_to_entry_map[best_match]
            item["corrected_parties"] = canonical_entry

# ------------------------------------------------------------------
# 4. Save the updated JSON back to disk (with the new keys)
# ------------------------------------------------------------------
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(data_json, f, ensure_ascii=False, indent=4)
